# NHL Draft Analysis - Determining the Performance Score

## Setup

In [5]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect("../data/nhl_drafts_clean.db")

print("Connected!")

Connected!


In [6]:
# Load full dataset
df = pd.read_sql_query("""
    SELECT draft_year, round, pick_number, drafted_by, player_name, 
           position, games_played, goals, assists, points, penalty_minutes
    FROM nhl_draft_picks
    ORDER BY draft_year, pick_number
""", conn)

# Replace null games_played and points with 0 for players who never reached NHL
df['games_played'] = df['games_played'].fillna(0)
df['points'] = df['points'].fillna(0)

print(f"Loaded {len(df)} players")
df.head()

Loaded 2766 players


,draft_year,round,pick_number,drafted_by,player_name,position,games_played,goals,assists,points,penalty_minutes
0,2005,1,1,Pittsburgh,Sidney Crosby,C,1408.0,652.0,1094.0,1746.0,892.0
1,2005,1,2,Anaheim,Bobby Ryan,R,866.0,261.0,308.0,569.0,470.0
2,2005,1,3,Carolina,Jack Johnson,D,1228.0,77.0,265.0,342.0,639.0
3,2005,1,4,Minnesota,Benoit Pouliot,L,625.0,130.0,133.0,263.0,371.0
4,2005,1,5,Montreal,Carey Price,G,712.0,0.0,13.0,13.0,51.0


## The Formula

To rank players within each draft in hindsight, we need a single metric 
that captures NHL effectiveness. The two available statistics that best 
reflect this are career points and career games played.

Points alone would favor offensive players and systematically undervalue 
defensemen and reliable depth players who provide value beyond scoring. 
Games played alone would fail to distinguish franchise cornerstones from 
journeymen. The formula combines both:

**Performance Score = games_played + (points * X)**

The coefficient X controls how much weight points carry relative to games 
played. A higher X pushes the rankings toward pure offensive production. 
A lower X gives more credit to longevity and consistency.

The goal is not perfect positional equality — in reality, elite forwards 
do score more points than elite defensemen, and that should be reflected. 
The goal is simply to find a value of X where the top of each draft's 
hindsight ranking passes the eye test, and where defensemen who are 
genuinely great are not unfairly buried.

**Note: Goalies are excluded from this ranking entirely, as their 
effectiveness cannot be meaningfully captured by a points and games 
played metric. Goalie draft performance is acknowledged as a limitation 
of this methodology.**

We test X = 1.0, 1.5, 2.0, and 2.5 against three benchmark drafts — 
2005, 2007, and 2015 — chosen because they contain well known players 
whose career trajectories are easy to evaluate intuitively.

## The Functions

In [42]:
def show_hindsight_topN(df, year, X, top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']
    df['performance_score'] = df['games_played'] + (df['points'] * X)
    df['gp_contribution'] = df['games_played']
    df['pts_contribution'] = (df['points'] * X).round(0)
    df['hindsight_rank'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')
    
    test = df[df['draft_year'] == year].sort_values('hindsight_rank').head(top_n)
    result = test[['hindsight_rank', 'pick_number', 'player_name', 'position',
                   'games_played', 'points', 'gp_contribution', 'pts_contribution',
                   'performance_score']].reset_index(drop=True)
    
    print(f"X = {X} | {year} Draft — Top {top_n} Hindsight Ranking")
    print(result.to_string())
    # display(result.style.set_caption(f"X = {X} | {year} Draft — Top {top_n} Hindsight Ranking")
    #                 .format({'hindsight_rank': '{:.0f}',
    #                          'games_played': '{:.0f}',
    #                          'points': '{:.0f}',
    #                          'gp_contribution': '{:.0f}',
    #                          'pts_contribution': '{:.0f}',
    #                          'performance_score': '{:.0f}'}))

In [52]:
def show_position_breakdown(df, x_values, top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']
    
    # Baseline — actual top N picks positional breakdown
    top_n_actual = df[df['pick_number'] <= top_n].copy()
    top_n_actual['position_group'] = top_n_actual['position'].apply(
        lambda x: 'D' if x == 'D' else 'F'
    )
    baseline = top_n_actual.groupby('position_group').size().reset_index(name='count')
    baseline['percentage'] = (baseline['count'] / len(top_n_actual) * 100).round(1)
    baseline = baseline.set_index('position_group')['percentage']
    
    print(f"Positional breakdown of hindsight top {top_n} vs actual top {top_n} picks\n")
    print(f"{'X Value':<12} {'F %':<10} {'D %':<10} {'F % actual':<15} {'D % actual':<15} {'F diff':<10} {'D diff'}")
    print("-" * 85)
    
    for X in x_values:
        df['performance_score'] = df['games_played'] + (df['points'] * X)
        df['hindsight_rank'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')
        
        top_n_df = df[df['hindsight_rank'] <= top_n].copy()
        top_n_df['position_group'] = top_n_df['position'].apply(
            lambda x: 'D' if x == 'D' else 'F'
        )
        
        breakdown = top_n_df.groupby('position_group').size().reset_index(name='count')
        total = breakdown['count'].sum()
        breakdown['percentage'] = (breakdown['count'] / total * 100).round(1)
        breakdown = breakdown.set_index('position_group')['percentage']
        
        f_pct = breakdown.get('F', 0)
        d_pct = breakdown.get('D', 0)
        f_actual = baseline.get('F', 0)
        d_actual = baseline.get('D', 0)
        f_diff = round(f_pct - f_actual, 1)
        d_diff = round(d_pct - d_actual, 1)
        
        print(f"X={X:<10} {f_pct:<10} {d_pct:<10} {f_actual:<15} {d_actual:<15} {f_diff:<10} {d_diff}")

In [45]:
def show_rank_movement(df, year, x_values=[1.0, 1.5, 2.0, 2.5], top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']
    
    # Calculate ranks for each X value
    rank_data = {}
    for X in x_values:
        df['performance_score'] = df['games_played'] + (df['points'] * X)
        df[f'rank_X{X}'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')
        rank_data[X] = df[df['draft_year'] == year][['player_name', 'position', 'pick_number', f'rank_X{X}']].copy()
    
    # Merge all ranks together on player_name
    merged = rank_data[x_values[0]][['player_name', 'position', 'pick_number', f'rank_X{x_values[0]}']].copy()
    for X in x_values[1:]:
        merged = merged.merge(rank_data[X][['player_name', f'rank_X{X}']], on='player_name')
    
    # Only keep players who appear in top N for at least one X value
    rank_cols = [f'rank_X{X}' for X in x_values]
    merged = merged[merged[rank_cols].min(axis=1) <= top_n]
    
    # Calculate max movement across all X values
    merged['max_movement'] = merged[rank_cols].max(axis=1) - merged[rank_cols].min(axis=1)
    
    # Sort by average rank
    merged['avg_rank'] = merged[rank_cols].mean(axis=1)
    merged = merged.sort_values('avg_rank')
    
    # Rename columns for readability
    merged.columns = ['player', 'pos', 'actual_pick'] + [f'X={X}' for X in x_values] + ['max_movement', 'avg_rank']
    merged = merged.reset_index(drop=True)
    
    print(f"{year} Draft — Rank Movement Across Coefficients (top {top_n})")
    print(merged.to_string())
    # display(merged.style.set_caption(f"{year} Draft — Rank Movement Across Coefficients")
    #                 .format({col: '{:.0f}' for col in merged.columns if col not in ['player', 'pos']}))

The three functions above are the tools used throughout this investigation.

`show_hindsight_topN` shows the top N players from a given draft year at 
a given coefficient, including a breakdown of how each component contributes 
to the final score.

`show_position_breakdown` 
tracks the forward/defenseman split within the top 30 across all drafts 
for a given coefficient, providing a quantitative check on positional fairness.

`show_rank_movement` shows every player who appears in the top 30 for any 
value of X, with their rank at each coefficient side by side. The 
`max_movement` column highlights which players are most sensitive to the 
choice of coefficient, and the `avg_rank` column is used to sort the table 
naturally from best to worst.

## Coefficient Investigation

Rather than arbitrarily selecting values of X to test, we first derive a 
natural anchor point from the data itself — the value of X at which games 
played and points contribute equally to the performance score.

For two components to contribute equally, their average contributions must 
be equal. That means:

**avg_games_played = avg_points * X**

Solving for X gives us:

**X = avg_games_played / avg_points**

We calculate this from the data below, using only players who reached the NHL.

In [47]:
# Only players who reached the NHL
nhl_players = df[(df['games_played'] > 0) & (df['points'] > 0)]
avg_gp = nhl_players['games_played'].mean()
avg_pts = nhl_players['points'].mean()
natural_X = avg_gp / avg_pts
print(f"Average GP: {avg_gp:.1f}")
print(f"Average Points: {avg_pts:.1f}")
print(f"Natural equal-weight X: {natural_X:.2f}")

Average GP: 367.4
Average Points: 160.2
Natural equal-weight X: 2.29


With a natural equal-weight X of 2.29, we now construct a symmetric set of 
test values centered on this anchor. We use a gap of 1.0 on each side, 
giving us five values that span from heavily favoring longevity to heavily 
favoring scoring:

| X Value | Interpretation |
|---------|----------------|
| 0.29 | Heavily favors longevity |
| 1.29 | Moderately favors longevity |
| 2.29 | Equal weighting |
| 3.29 | Moderately favors scoring |
| 4.29 | Heavily favors scoring |

We test each value against three benchmark drafts — 2005, 2007, and 2015 — 
chosen because they contain well known players whose career trajectories 
are easy to evaluate intuitively. The `show_rank_movement` function shows 
how player rankings shift as X changes, with the `max_movement` column 
highlighting which players are most sensitive to the choice of coefficient.

In [48]:
natural_X = round(natural_X, 2)
gap = 1.0
x_values = [round(natural_X - 2*gap, 2),
            round(natural_X - gap, 2),
            natural_X,
            round(natural_X + gap, 2),
            round(natural_X + 2*gap, 2)]

print(f"Test values: {x_values}")

Test values: [0.29, 1.29, 2.29, 3.29, 4.29]


In [50]:
show_rank_movement(df, 2005, x_values=x_values)
show_rank_movement(df, 2007, x_values=x_values)
show_rank_movement(df, 2015, x_values=x_values)

2005 Draft — Rank Movement Across Coefficients (top 30)
                   player pos  actual_pick  X=0.29  X=1.29  X=2.29  X=3.29  X=4.29  max_movement  avg_rank
0           Sidney Crosby   C            1     1.0     1.0     1.0     1.0     1.0           0.0       1.0
1            Anze Kopitar   C           11     2.0     2.0     2.0     2.0     2.0           0.0       2.0
2       Kristopher Letang   D           62     3.0     3.0     3.0     4.0     4.0           1.0       3.4
3            Paul Stastny   C           44     6.0     4.0     4.0     3.0     3.0           3.0       4.0
4              T.J. Oshie   R           24     9.0     6.0     5.0     5.0     5.0           4.0       6.0
5            Keith Yandle   D          105     8.0     5.0     6.0     6.0     6.0           3.0       6.2
6         Andrew Cogliano   C           25     5.0     7.0     7.0     7.0     8.0           3.0       6.8
7     Marc-Edouard Vlasic   D           35     4.0     8.0     8.0    11.0    11.0      

### Observations — Rank Movement

The top 5-7 players in each benchmark draft are remarkably stable across 
all five coefficients, suggesting the methodology is robust at identifying 
the clear standout players from each class. The most notable movers are 
players at the extremes of the games played vs points spectrum.

In the 2015 draft, Kirill Kaprizov (pick 135) shows the largest movement 
of any player across all three drafts — rising from 33rd at X=0.29 to 
13th at X=4.29 as his elite scoring rate is increasingly rewarded. In the 
opposite direction, Brandon Carlo (pick 37) falls 15 spots as X increases, 
reflecting his profile as a reliable defensive defenseman with modest 
point totals. These movers illustrate the core tension the coefficient 
is balancing — longevity and reliability vs offensive production.

In [53]:
show_position_breakdown(df, x_values)

Positional breakdown of hindsight top 30 vs actual top 30 picks

X Value      F %        D %        F % actual      D % actual      F diff     D diff
-------------------------------------------------------------------------------------
X=0.29       69.2       30.8       66.8            33.2            2.4        -2.4
X=1.29       69.7       30.3       66.8            33.2            2.9        -2.9
X=2.29       70.8       29.2       66.8            33.2            4.0        -4.0
X=3.29       71.5       28.5       66.8            33.2            4.7        -4.7
X=4.29       72.1       27.9       66.8            33.2            5.3        -5.3


### Observations — Positional Fairness

We evaluate positional fairness across the hindsight top 30. The top 30 
is the most meaningful cut because it represents roughly one pick per 
team per draft — the most scrutinized selection each team makes. Going 
higher starts to include players where the signal gets noisy and the 
stakes of each individual pick are lower.

The hindsight top 30 exhibits a mild forward bias across all values of X 
that increases gradually as X rises. At our chosen equal-weight value of 
X=2.29, forwards are over-represented by 4.0 percentage points relative 
to the actual top 30 picks. Across all 13 drafts in the dataset this 
equates to roughly 16 additional forwards in the hindsight top 30 — just 
over one extra forward per draft class. This is acknowledged as a known 
limitation of the methodology.

Notably, even at X=0.29 where games played dominates heavily, a 2.4% 
forward bias persists. This suggests the bias is not purely a product 
of the formula but partially reflects the reality that forwards who reach 
the NHL tend to accumulate more games played as well as more points. 
No value of X eliminates the bias entirely. The bias is at its smallest 
at the lower end of the range and grows gradually as X increases, 
suggesting that a lower value of X is preferable from a positional 
fairness standpoint.

## The Decision

Both lines of evidence point in the same direction. The rank movement 
analysis shows the top of each draft's hindsight ranking is stable across 
all five coefficients — the methodology is robust regardless of which 
value of X is chosen. The positional fairness analysis shows the forward 
bias is smallest at the lower end of the range and grows as X increases.

X = **2.29** is selected as the final coefficient. This value is derived 
directly from the data as the natural equal-weight point between games 
played and points — the value at which both components contribute equally 
to the performance score on average. It makes no philosophical assumptions 
about whether longevity or scoring is more valuable in a draft pick. And 
sitting at the middle of our tested range, it keeps the forward 
bias at a modest level while still meaningfully rewarding offensive 
production.

The performance score formula is therefore:

**Performance Score = games_played + (points * 2.29)**

This formula will be used to build the hindsight ranking for every draft 
from 2005 to 2017, against which each team's actual selections will be 
compared to produce the final draft efficiency scores.

## Applying the Formula

## End of Analysis